## Step 1: OCR ##
The first step is to download paddleOCR and test it on some of the dataset. PaddleOCR is ran with *PaddlePaddle* as their deep learning model so it's important to download both libraries together

Source: https://github.com/PaddlePaddle/PaddleOCR

For CPU-only PaddlePaddle:
!python -m pip install paddlepaddle==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/

In [1]:
from paddleocr import PaddleOCR

c:\Users\alexr\anaconda3\envs\MLP\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
c:\Users\alexr\anaconda3\envs\MLP\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.


In [ ]:
ocr = PaddleOCR(use_doc_orientation_classify=False, use_doc_unwarping=False, use_textline_orientation=False)
result = ocr.predict(input="data/electric bills-part-1.pdf")

for page in result:
  page.save_to_img("output")
  page.save_to_json("output")


Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\alexr\.paddlex\official_models\PP-OCRv5_server_det`.
Creating model: ('PP-OCRv5_server_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\alexr\.paddlex\official_models\PP-OCRv5_server_rec`.


In [12]:
import os, json, re

OUTPUT_DIR = "output"

num_re = re.compile(r"^-?\d+(\.\d+)?$")

def is_numeric_token(tok):
    # strip common non-numeric chars (currency, commas, percent, parentheses)
    cleaned = re.sub(r"[^\d\.\-]", "", tok)
    return bool(cleaned) and bool(num_re.match(cleaned))

def extract_text_lines(data):
    if isinstance(data, dict):
        return data.get("rec_texts") or data.get("texts") or []
    if isinstance(data, list):
        # maybe list of [box, text] or [[x1,y1,...], "text"]
        texts = []
        for it in data:
            if isinstance(it, list) and len(it) >= 2 and isinstance(it[-1], str):
                texts.append(it[-1])
        return texts
    return []

for fname in sorted(os.listdir(OUTPUT_DIR)):
    if not fname.endswith("_res.json"):
        continue
    path = os.path.join(OUTPUT_DIR, fname)
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    lines = extract_text_lines(data)
    total_words = 0
    numeric_words = 0

    for line in lines:
        # simple tokenization on whitespace; adjust if you need finer splitting
        tokens = re.findall(r"\S+", line)
        total_words += len(tokens)
        for t in tokens:
            if is_numeric_token(t):
                numeric_words += 1

    print(f"{fname}: lines={len(lines)}, total_words={total_words}, numeric_words={numeric_words}")

electric bills-part-1_0_res.json: lines=114, total_words=281, numeric_words=49
electric bills-part-1_1_res.json: lines=120, total_words=589, numeric_words=54
electric bills-part-1_2_res.json: lines=18, total_words=92, numeric_words=3
